# Smoke test — everything, end to end

Pushes **every model** through dense → iterative pruning (magnitude @ 0.95) → BaCP, at 2 epochs on 2 batches. The numbers are meaningless by construction; what this proves is that every pipeline runs on THIS machine: weights load (preflight), data loads, all three scripts train, and a run record lands for each phase. Records carry a `.smoke` key suffix so they can never satisfy a real experiment.

Run the cells in order. Each model is one cell; the last cell is the PASS/FAIL table. A model that is already smoke-tested is skipped — delete its records under `results/runs/` to re-test.

CV models use CIFAR-10; the language models use SST-2 (GLUE test split is unlabelled, so "test" accuracy is the validation set).

In [ ]:
import sys, pathlib

# Find nb_common.py whether the kernel started in this folder or at the repo root.
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

import nb_common as nb
info = nb.setup()

In [ ]:
GPU = 0
SEED = 1
outcomes = {}

def smoke(model):
    """Dense -> prune -> bacp for one model, 2 epochs x 2 batches each."""
    results = []
    for phase, kw in [('dense', {}),
                      ('prune', dict(pruner='magnitude', sparsity=0.95)),
                      ('bacp',  dict(pruner='magnitude', sparsity=0.95))]:
        cell = nb.make_cell(model, phase, seed=SEED, smoke=True, **kw)
        try:
            out = nb.run(cell, gpu=GPU)
            ok = out is None or cell['key'] in __import__('runner').completed_keys()
            results.append((phase, 'PASS' if ok else 'FAIL (no record)'))
        except Exception as exc:
            results.append((phase, f'FAIL ({type(exc).__name__}: {exc})'))
            break   # later phases need the earlier checkpoint
    outcomes[model] = results
    for phase, verdict in results:
        print(f'  => {model:<28} {phase:<6} {verdict}')

## resnet34

Dense → magnitude I.P. → BaCP, smoke-sized.

In [ ]:
nb.fetch_imagenet_weights('resnet34')
nb.preflight('resnet34', num_classes=nb.FAMILIES['resnet34']['base']['num_classes'])
smoke('resnet34')

## resnet50

Dense → magnitude I.P. → BaCP, smoke-sized.

In [ ]:
nb.fetch_imagenet_weights('resnet50')
nb.preflight('resnet50', num_classes=nb.FAMILIES['resnet50']['base']['num_classes'])
smoke('resnet50')

## vgg11

Dense → magnitude I.P. → BaCP, smoke-sized.

In [ ]:
nb.fetch_imagenet_weights('vgg11')
nb.preflight('vgg11', num_classes=nb.FAMILIES['vgg11']['base']['num_classes'])
smoke('vgg11')

## vgg19

Dense → magnitude I.P. → BaCP, smoke-sized.

In [ ]:
nb.fetch_imagenet_weights('vgg19')
nb.preflight('vgg19', num_classes=nb.FAMILIES['vgg19']['base']['num_classes'])
smoke('vgg19')

## vit-tiny

Dense → magnitude I.P. → BaCP, smoke-sized. 224px inputs — the slowest smoke here.

In [ ]:
nb.preflight('vit-tiny', num_classes=nb.FAMILIES['vit-tiny']['base']['num_classes'])
smoke('vit-tiny')

## vit-small

Dense → magnitude I.P. → BaCP, smoke-sized. 224px inputs — the slowest smoke here.

In [ ]:
nb.preflight('vit-small', num_classes=nb.FAMILIES['vit-small']['base']['num_classes'])
smoke('vit-small')

## distilbert-base-uncased

Dense → magnitude I.P. → BaCP, smoke-sized. First run downloads the model and SST-2.

In [ ]:
nb.preflight('distilbert-base-uncased', num_classes=nb.FAMILIES['distilbert-base-uncased']['base']['num_classes'])
smoke('distilbert-base-uncased')

## roberta-base

Dense → magnitude I.P. → BaCP, smoke-sized. First run downloads the model and SST-2.

In [ ]:
nb.preflight('roberta-base', num_classes=nb.FAMILIES['roberta-base']['base']['num_classes'])
smoke('roberta-base')

## Verdict

Every row must read PASS. A FAIL names its phase and error; fix that before spending real GPU time in the per-model notebooks.

In [ ]:
print(f'{"model":<28} {"dense":<8} {"prune":<8} {"bacp":<8}')
all_ok = True
for model, results in outcomes.items():
    row = {phase: verdict for phase, verdict in results}
    cells_ = [row.get(p, '-') for p in ('dense', 'prune', 'bacp')]
    all_ok &= all(v == 'PASS' for v in cells_ if v != '-') and len(results) == 3
    print(f'{model:<28} ' + ' '.join(f'{c:<8}' for c in cells_))
print()
print('ALL PIPELINES PASS' if all_ok and outcomes else 'SOMETHING FAILED -- read the cell above with the failure')